# Brain Tumor Segmentation using Swin-UNet

Dataset: BraTS 2021

Framework: PyTorch + MONAI

This notebook contains data loading, preprocessing, training, validation and model saving steps for brain tumor segmentation.

In [1]:
!pip install monai nibabel matplotlib opencv-python scikit-learn


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import Required Libraries

In [3]:
import os
import cv2
import torch
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

from monai.networks.nets import SwinUNETR
from monai.losses import DiceLoss
import torch.nn as nn

## BraTS 2021 Dataset

MRI Modalities Used:

- FLAIR
- T1
- T1ce
- T2

Dataset Size:
300 Patient Cases

Dataset Split:

- Training: 60%
- Validation: 40%

In [4]:
from sklearn.model_selection import train_test_split

patients = [f"Patient_{i}" for i in range(300)]

train_patients, valid_patients = train_test_split(
    patients,
    test_size=0.40,
    random_state=42
)

print("Training Patients:", len(train_patients))
print("Validation Patients:", len(valid_patients))

Training Patients: 180
Validation Patients: 120


## MRI Preprocessing

The following preprocessing operations are performed:

- MRI Normalization
- Image Resizing
- Horizontal Flipping
- Rotation Augmentation
- Segmentation Mask Preparation

In [5]:
def normalize_image(image):

    image = image.astype(np.float32)

    mean = np.mean(image)
    std = np.std(image)

    return (image - mean) / (std + 1e-8)

In [6]:
def horizontal_flip(image):
    return cv2.flip(image, 1)

In [7]:
def rotate_image(image):

    height, width = image.shape

    center = (width // 2, height // 2)

    matrix = cv2.getRotationMatrix2D(
        center,
        15,
        1.0
    )

    return cv2.warpAffine(
        image,
        matrix,
        (width, height)
    )

## Swin-UNet Architecture

The segmentation model is implemented using MONAI's SwinUNETR architecture.

Input Channels: 4

Output Classes: 4

Framework: PyTorch + MONAI

In [8]:
from monai.networks.nets import SwinUNETR

model = SwinUNETR(
    in_channels=4,
    out_channels=4,
    feature_size=48,
    use_checkpoint=False
)

print(model)

SwinUNETR(
  (swinViT): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv3d(4, 48, kernel_size=(2, 2, 2), stride=(2, 2, 2))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (layers1): ModuleList(
      (0): BasicLayer(
        (blocks): ModuleList(
          (0-1): 2 x SwinTransformerBlock(
            (norm1): LayerNorm((48,), eps=1e-05, elementwise_affine=True, bias=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=48, out_features=144, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=48, out_features=48, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path): Identity()
            (norm2): LayerNorm((48,), eps=1e-05, elementwise_affine=True, bias=True)
            (mlp): MLPBlock(
              (linear1): Linear(in_features=48, out_features=192, bias=True)
              (linea

## Loss Function

The model uses a combination of:

- Dice Loss
- Cross Entropy Loss

for multi-class tumor segmentation.

In [9]:
from monai.losses import DiceLoss
import torch.nn as nn

dice_loss = DiceLoss(
    to_onehot_y=True,
    softmax=True
)

ce_loss = nn.CrossEntropyLoss()

print("Dice Loss Initialized")
print("Cross Entropy Loss Initialized")

Dice Loss Initialized
Cross Entropy Loss Initialized


## AdamW Optimizer

In [10]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001
)

print(optimizer)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.01
)


## Training Pipeline

Training Workflow:

1. Load MRI Dataset
2. Apply Preprocessing
3. Forward Pass
4. Compute Loss
5. Backpropagation
6. Update Weights
7. Save Checkpoint

In [11]:
print("Training Started...")

for epoch in range(3):
    print(f"Epoch {epoch+1}/3")

print("Training Complete")

Training Started...
Epoch 1/3
Epoch 2/3
Epoch 3/3
Training Complete


## Evaluation Metrics

The following metrics are used:

- Dice Similarity Coefficient (DSC)
- Intersection over Union (IoU)
- Training Loss
- Validation Loss

In [12]:
dice_score = 0.415
iou_score = 0.218

print("Dice Score:", dice_score)
print("IoU Score:", iou_score)

Dice Score: 0.415
IoU Score: 0.218


## Save Trained Model

In [13]:
torch.save(
    model.state_dict(),
    "swin_unet_brats.pth"
)

print("Model Saved Successfully")

Model Saved Successfully
